# Lesson 69 · The CTC forward algorithm — packing exponentially many paths into one table, in pure NumPy

**Learning goals**
- Understand CTC's core challenge: we don't know which frame each character is aligned to
- Master the recursive definition of the forward variable α (alpha)
- Implement the CTC forward algorithm in pure NumPy and verify it numerically

> **Same notation** as L68; the numbers usually live in the log domain to avoid underflow.

← **Previous lesson** · [L68 · How CTC alignment works](L68_ctc_alignment.ipynb)

> Last lesson covered **how CTC alignment works**: the blank symbol, monotonic paths and label collapsing.
> This lesson explores **the CTC forward algorithm**.

## This lesson's story: why does naive path enumeration "explode"?

Last lesson built the intuition for CTC: the blank symbol lets one target sequence correspond to exponentially many paths. This lesson squeezes that "explosion" back into a single table.

But here's the problem: **to compute P("ab" | 6 frames) exactly, we'd have to enumerate 3⁶ = 729 paths, collapse and check each one, then add them up**. At T = 100 frames that's 3¹⁰⁰ paths — completely hopeless.

The **forward algorithm**'s answer: dynamic programming, turning "enumerate → sum" into "recurse → merge".

**The key design**: expand the target sequence `["a","b"]` into a sequence that includes blanks, `l' = [∅, a, ∅, b, ∅]` (a blank before and after every character), of length `S = 2L+1`.

**The forward variable** α[t][s] = "the total probability of all valid paths that, at the end of frame t, finish on l'[s]".

**The recursion (two cases)**:
```
Case 1: l'[s] = blank, or the same character as two steps back
   α[t][s] = (α[t-1][s] + α[t-1][s-1]) × P(l'[s] | t)

Case 2: l'[s] = a character, and l'[s] ≠ l'[s-2]
   α[t][s] = (α[t-1][s] + α[t-1][s-1] + α[t-1][s-2]) × P(l'[s] | t)
```

Finally: `P(label | logits) = α[T-1][S-1] + α[T-1][S-2]` — on the last frame a path may stop on the trailing blank or on the last character; in a log-domain implementation this addition is done with `logaddexp`.

The cost drops from exponential to **O(T·S)**, where `S = 2L + 1` — and that is what makes training ASR possible.

### What is a "valid path"?

Imagine climbing a staircase. Each frame is one step forward in time, and each position is a "height" on the stairs. A **valid path** is one way of climbing:
- **Start at the bottom**: frame 0 must begin at position 0 (blank) or position 1 (the first character).
- **Climb or stay put**: each step moves up at most one stair (position s → s or s+1), or **skips one stair** (position s → s+2), but only in particular situations.
- **Why not jump to s+3?** If arbitrarily large jumps were allowed, a single label could be reached in far too many ways, and DP could no longer merge the work efficiently. The key constraint is that **positions on adjacent frames differ by at most 2**, which is what lets the whole thing finish in O(T·S).

**Why does the "skip two" move need the extra check `l'[s] ≠ l'[s-2]`?**

To avoid **counting the wrong paths**. Consider this example:
- The target label is `[a]`, expanded to `l' = [∅, a, ∅]` (positions 0, 1, 2)
- One path: frame 0 sits at position 0 (∅), frame 1 at position 1 (a)
  - This path "moves" from position 0 to position 1 (one step)
- Another path: frame 0 sits at position 1 (a), frame 1 at position 1 (a)
  - This path stays at position 1 (zero steps, i.e. staying put)

Neither of these needs a two-step skip. What matters is the case where `l'[s]` and `l'[s-2]` are **the same character** (say both 'a'):
- Jumping straight from `l'[s-2]` to `l'[s]` would "skip the blank in between"
- But that blank is exactly what separates two identical characters. Skip it and, when the path is collapsed, "a a" merges into a single "a" — so we would wrongly count a path for "a" as if it were a path for "a-blank-a", i.e. two a's

So the condition `l'[s] ≠ l'[s-2]` guarantees that **we may skip the blank in between only when the character we jump from differs from the one we land on**. That's what keeps the collapsing rule working correctly.


In [ ]:
import numpy as np

In [ ]:
# Build a toy example: 6 frames, vocabulary {a:0, b:1, blank:2}, target 'ab'
BLANK = 2
VOCAB_SIZE = 3
T = 6
LABELS = [0, 1]   # 'a', 'b'

np.random.seed(42)
logits = np.random.randn(T, VOCAB_SIZE)
# convert to log-probabilities
# numerically stable log-softmax: subtract the max first so exp can't overflow
logits_s = logits - logits.max(axis=1, keepdims=True)
log_probs = logits_s - np.log(np.exp(logits_s).sum(axis=1, keepdims=True))
print(f'log_probs shape: {log_probs.shape}  (T={T}, vocab={VOCAB_SIZE})')

## 🔬 First, fill in a tiny α table by hand (T=3, label "AB")

The first time you meet CTC's DP, forget about 100 frames. With **T=3 and target `"AB"`**, fill in three rows of α by hand and walk the recursion through once — after that the code won't look scary.

**Expanded label** `l' = [_, A, _, B, _]`, positions `s = 0..4` (`S = 2L+1 = 5`).
`α[t][s]` = "the total probability of all valid paths that reach frame t sitting on `l'[s]`". To see the structure, the table below only marks **which cells are non-zero** (`·` = unreachable, still -inf):

```
 s=      0(_)   1(A)   2(_)   3(B)   4(_)
t=0       ✓      ✓      ·      ·      ·     start: only the leading _ or the first character A
t=1       ✓      ✓      ✓      ✓      ·     at most one blank skipped per step, so B (s=3) is just reachable now
t=2       ·      ✓      ✓      ✓      ✓     to finish "AB" in 3 frames, we must end on B (s=3) or the closing _ (s=4)
```

**Which predecessors feed each cell** (these are the three `logaddexp` branches in the TODO):
- Stay put: `α[t-1][s]`
- Advance one cell from the left: `α[t-1][s-1]`
- **Skip one blank**: `α[t-1][s-2]`, **only if** `l'[s]` is a character and `l'[s] ≠ l'[s-2]` (so `A_A` is never mistaken for a single A)

**Final answer** = `logaddexp(α[2][4], α[2][3])` — on the last frame a path may stop on the closing `_` or on the last character `B`.

Keep the "which cells are reachable" table above in view as you read the implementation below: the three `if`s are just describing "only move right, skip at most one blank".


### Intuition for the initial and final states

**Why can frame 0 start only from the first two positions?**

The target `[a, b]` expands to `[∅, a, ∅, b, ∅]`. At frame 0 nothing has been emitted yet, so there are only two ways to open:
1. **Stay on blank**: sit at position 0 (the leading ∅) — "frame 0 emits a blank; the characters haven't arrived yet"
2. **Emit the first character**: sit at position 1 (the first a) — "frame 0 emits a straight away"

Any other position would mean "crossing several characters within one frame", which breaks monotonicity. So:
```
α[0, 0] = log P(∅ | frame_0)
α[0, 1] = log P(a | frame_0)
α[0, 2] = α[0, 3] = ... = -∞    # unreachable
```

**Why is the answer on the last frame a logaddexp of two positions?**

On frame T-1 (the last frame) there are two valid final states:
1. **Stop on the last character**: position S-2 (the final b)
   - "the last frame emits b, and the path ends"
2. **Stop on the trailing blank**: position S-1 (the final ∅)
   - "the characters were all emitted on earlier frames; the last frame is blank padding"

The probabilities of both final states must be added, because both collapse to the target label `[a, b]`. Position S-3 (the blank between a and b) and anything earlier can't be a final state, because from there the path never emits b.

So the answer = α[T-1][S-1] + α[T-1][S-2] (with logaddexp in the log domain).


### Building the expanded label l', and how the indices line up

**Example: original labels=[0, 1] (meaning 'a'=0, 'b'=1), blank=2**

```
How it's built:
  1. Initialize: lprime = [blank] = [2]
  2. Loop over the labels:
     - append labels[0]=0: lprime = [2, 0]
     - append blank:       lprime = [2, 0, 2]
     - append labels[1]=1: lprime = [2, 0, 2, 1]
     - append blank:       lprime = [2, 0, 2, 1, 2]

Expanded label: l' = [2, 0, 2, 1, 2]
                      ↓  ↓  ↓  ↓  ↓
                      ∅  a  ∅  b  ∅
Position index: s=    0  1  2  3  4

Length S = 2L+1 = 2*2+1 = 5
```

**When we check `l'[s] != l'[s-2]`:**
- s=0: no s-2, skip
- s=1: no s-2, skip
- s=2: l'[2]=2 (∅), l'[0]=2 (∅) → equal, no jump from s-2
- s=3: l'[3]=1 (b), l'[1]=0 (a) → different, jump from s-2 **allowed**
- s=4: l'[4]=2 (∅), l'[2]=2 (∅) → equal, no jump from s-2

**Key observation**: this condition makes sure that "the blank in between may be skipped only when the target character differs from the one two steps back". That way "a-blank-a" is never wrongly squashed into "a".


## ✏️ Implement the CTC forward algorithm

**Signature**:
```python
def ctc_forward(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    # log_probs: (T, V) matrix of log-probabilities
    # labels: list[int] target sequence (without blanks)
    # returns: float  log P(labels | log_probs)
```

**A 5-step implementation plan**:

| Step | What to do | Key point |
|---|---|---|
| 1 | Build the extended label `l'` | `[blank, L[0], blank, L[1], ..., blank]`, length `2L+1` |
| 2 | Initialize α to `-inf`, then set `α[0][0]` and `α[0][1]` | Frame 0 can only be blank or the first character |
| 3 | For t=1..T-1: recurse over s=0..S-1 | Case 1 (blank/repeat) takes 2 terms, case 2 takes 3 |
| 4 | Use log-space addition to avoid underflow | `logsumexp(a, b) = a + log(1 + exp(b-a))` |
| 5 | Return `logsumexp(α[T-1][S-1], α[T-1][S-2])` | The total probability of the two ways to finish |

**Acceptance criteria**:
- Matches brute-force enumeration (`ctc_forward_brute_force`) to within 1e-5
- Returns a finite float (not -inf) for `T=6, L=["a","b"]`

**One thing to note before you start**: `S = 2L + 1`, and this lesson's complexity is `O(T·S)` — not `O(T×2L)`.


### Edge details in the implementation

**Index checks**:
- `if s > 0`: makes sure s-1 ≥ 0, allowing a move from one cell to the left
- `if s > 1`: makes sure s-2 ≥ 0 and the "different character" condition holds, allowing a jump from two cells to the left

**Why do we return alpha[-1, -1] and alpha[-1, -2]?**
- `alpha[-1, -1]` = `alpha[T-1, S-1]`: last frame, last position (the trailing blank)
- `alpha[-1, -2]` = `alpha[T-1, S-2]`: last frame, second-to-last position (the last character of the target label)
- These two final states cover every way a valid path can end: either on the last character or on the trailing blank

**Why can't we take only alpha[-1, -1]?**
- Taking only the trailing blank would throw away every path whose final frame is still emitting the last character (e.g. "…a b b" with no blank at the end)
- Those paths end in state S-2 (the last character), and their probability must not be dropped

**Logic flow** (with labels=[a,b], T=3 as the example):
```
t=0: [✓, ✓, ·, ·, ·]     start: only positions 0 and 1 are reachable
     ↓
t=1: [✓, ✓, ✓, ✓, ·]     recursion: each frame advances by at most 1 or 2 positions
     ↓
t=2: [·, ✓, ✓, ✓, ✓]     final: positions 3 and 4 added together = the answer
```


### The log-domain equivalent (the implementation uses this)

```text
log α[t, s] = log p[t, l'[s]] + logaddexp(
    log α[t-1, s],      # stay
    log α[t-1, s-1],    # move one step
    log α[t-1, s-2]     # jump two steps (only if l'[s] ≠ l'[s-2])
)
```

`np.logaddexp(a, b)` = `log(exp(a) + exp(b))`, computed stably; three predecessors can be combined with two nested `logaddexp` calls.

### Why subtract the max? A concrete underflow crisis

Imagine you're holding 100 probabilities, each 0.8, and you have to multiply them all together:
```
P = 0.8^100
```

**What happens if you compute it directly?**
- In ordinary floating point: 0.8^100 ≈ 2.04×10^(-10) — already tiny, and every extra frame shrinks it further
- At 0.8^1000 ≈ 10^(-97) it is still representable, but a few thousand frames push it below the smallest positive double (about 10^-308) and it "underflows" to 0 — the program ends up with P = 0 (completely wrong)

**The log domain to the rescue:**
```
log(P) = log(0.8^100) = 100 × log(0.8) ≈ 100 × (-0.223) = -22.3
```
Forget about the size of the number and just keep track of the "exponent" — far more stable.

**But there's another trap: exp overflow**. If the logits themselves are large (say [1000, 1001, 999]), computing softmax directly starts with exp:
```
softmax = exp([1000, 1001, 999]) / sum(exp(...))
```
exp(1000) and exp(1001) are both far beyond the floating-point limit (about 10^308) — it blows up straight away!

**The fix is "subtract the max":**
```
logits_shifted = [1000, 1001, 999] - 1001 = [-1, 0, -2]
softmax = exp([-1, 0, -2]) / sum(exp(...))   # all values in a manageable range
log(softmax) = logits_shifted - log(sum(exp(logits_shifted)))
```
This avoids exp overflow, and the final result is exactly the same (the ratios between the probabilities haven't changed).

**How this shows up in CTC**:
- The input logits can be large (neural-network outputs) → stabilize with log-softmax
- Multiplying the probabilities along a path → becomes a sum in the log domain → add paths stably with logaddexp
- Pooling many paths at the end → one more logaddexp does it

The code below compares "compute it directly" with "the stable log version".


In [ ]:
# Numerical demo: concrete examples of underflow and overflow
import numpy as np

print("=" * 60)
print("[Example 1] Multiplying many probabilities → underflow")
print("=" * 60)

# simulate a 5000-frame path where every frame has probability 0.8 (0.8^100 doesn't underflow yet; it takes thousands of frames)
T = 5000
p_per_frame = 0.8
log_p_per_frame = np.log(p_per_frame)

# multiply directly in the probability domain
try:
    p_direct = p_per_frame ** T   # a power, not a running product: repeated ×0.8 gets stuck at the smallest subnormal ~1e-323 instead of reaching 0
    print(f"Direct: P = 0.8^{T} = {p_direct}")
    if p_direct == 0:
        print("  ⚠️  The result became 0 (underflowed to nothing)!")
except:
    print("❌ Computation failed (overflow or underflow)")

# add in the log domain
log_p_log_domain = T * log_p_per_frame
print(f"\nLog domain: log(P) = {T} × {log_p_per_frame:.4f} = {log_p_log_domain:.4f}")
print(f"As a power of 10: P ≈ 10^{log_p_log_domain / np.log(10):.1f} (exp back to a probability would still underflow, so stay in the log domain)")
print("  ✅ Numerically stable, sensible result")

print("\n" + "=" * 60)
print("[Example 2] Softmax of large logits → exp overflow")
print("=" * 60)

# simulate large neural-network outputs
logits = np.array([1000.0, 1001.0, 999.0])
print(f"Logits: {logits}")

# the wrong way: softmax directly
print("\n❌ Wrong way (exp directly):")
try:
    exp_logits = np.exp(logits)
    print(f"  exp(logits) = {exp_logits}")
    softmax_wrong = exp_logits / exp_logits.sum()
    print(f"  softmax = {softmax_wrong}")
    if np.any(np.isnan(softmax_wrong)) or np.any(np.isinf(softmax_wrong)):
        print("  ⚠️  NaN or Inf appeared (overflow)!")
except:
    print("  ❌ Computation failed (overflow)")

# the right way: subtract the max
print("\n✅ Right way (the log-softmax trick):")
m = logits.max()
logits_shifted = logits - m
print(f"  Logits - max = {logits_shifted}")
exp_shifted = np.exp(logits_shifted)
softmax_stable = exp_shifted / exp_shifted.sum()
print(f"  softmax = {softmax_stable}")
print(f"  log(softmax) = {np.log(softmax_stable)}")
print("  ✅ Numerically stable, no overflow")

print("\n" + "=" * 60)
print("[Example 3] Why logaddexp is stable")
print("=" * 60)

# two very small log-probabilities
log_p1 = -1000.0
log_p2 = -1001.0
print(f"log_p1 = {log_p1}, log_p2 = {log_p2}")

# the wrong way: add them directly (in the log domain that's multiplication, not addition!)
print(f"\n❌ Wrong way (log_p1 + log_p2 = {log_p1 + log_p2})")
print("  This actually does \"multiplication\" in the log domain, not the \"addition\" we want")

# the right way: logaddexp
log_sum = np.logaddexp(log_p1, log_p2)
print(f"\n✅ Right way (logaddexp):")
print(f"  log(exp(log_p1) + exp(log_p2)) = {log_sum}")
print(f"  Back to a probability: exp({log_sum}) ≈ {np.exp(log_sum):.2e}")
print("  ✅ Stably computed the \"addition\" (in the probability domain)")

In [ ]:
a, b = np.log([0.3, 0.7])
assert np.isclose(np.logaddexp(a, b), np.log(1.0))
print("✅ logaddexp demo: log(exp(a)+exp(b)) adds stably in the log domain")

### Why we need logaddexp: what does "addition" become in the log domain?

**The key realization: log is non-linear**

Suppose two different paths have probabilities p₁=0.3 and p₂=0.7, and we want p₁+p₂=1.0.

In the log domain:
```
log(p₁) ≈ -1.204
log(p₂) ≈ -0.357
```

**The wrong way**: add them directly
```
log(p₁) + log(p₂) ≈ -1.204 + (-0.357) = -1.561
```
That actually computes log(p₁ × p₂) (the log of a product), not log(p₁ + p₂)!

If we then took exp(-1.561) ≈ 0.21, we'd have a completely wrong answer (it should be 1.0).

**The right way**: logaddexp
```
logaddexp(log(p₁), log(p₂)) = log(exp(log(p₁)) + exp(log(p₂)))
                             = log(0.3 + 0.7)
                             = log(1.0)
                             = 0
```

**Intuition**:
- `log(a + b)` ≠ `log(a) + log(b)`, so we have to "go back" to probabilities, add, then take the log again
- NumPy's `logaddexp` is designed for exactly this, and does it efficiently and stably
- Three or more terms can be added by nesting calls: logaddexp(logaddexp(a, b), c)


In [ ]:
def log_softmax(logits, axis=-1):
    m = logits.max(axis=axis, keepdims=True)
    shifted = logits - m
    return shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))

In [ ]:
def ctc_forward(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    """CTC forward algorithm: compute log P(labels | log_probs).

    Args:
        log_probs: (T, vocab_size) log-probability matrix.
        labels:    list of integer label ids (without blanks).
        blank:     blank token index.
                   Note: in this notebook BLANK=2, so pass blank=BLANK explicitly;
                   the default of 0 is the CTC paper's convention and doesn't fit this example.

    Returns:
        log probability of the label sequence under CTC.
    """
    T, V = log_probs.shape
    L = len(labels)

    # extended label: blank + label + blank + label + ... + blank
    lprime = [blank]
    for c in labels:
        lprime.append(c)
        lprime.append(blank)
    S = len(lprime)   # = 2L + 1

    NEG_INF = -1e30
    alpha = np.full((T, S), NEG_INF)

    # ✏️ Step 1: initial conditions
    # Frame 0 can only start from the first two positions of the extended label:
    #   - position 0 (the leading blank): alpha[0, 0] = log_probs[0, lprime[0]]
    #   - position 1 (the first character): alpha[0, 1] = log_probs[0, lprime[1]] (when S > 1)
    # Every other position is unreachable at frame 0 (stays -inf)

    # ✏️ Step 2: the recursion for t=1..T-1
    # Each position s on each frame has up to three predecessor candidates:
    #   - stay: alpha[t-1, s]
    #   - come from one cell to the left: alpha[t-1, s-1] (when s > 0)
    #   - come from two cells to the left: alpha[t-1, s-2] (when s > 1 and lprime[s] != lprime[s-2];
    #     this condition keeps path collapsing correct, so "A-blank-A" is never mistaken for "A")
    # Merge the candidates with np.logaddexp in the log domain, then add log_probs[t, lprime[s]]

    # ✏️ Step 3: return the log-sum over the final states
    # The last frame may stop on position S-1 (the trailing blank) or S-2 (the last character);
    # the logaddexp of the two is the final answer

    raise NotImplementedError("implement ctc_forward (steps 1-3)")


## Half-step exercise B · ctc_forward_brute_force (write / read this function first)

First get clear on "what exactly we are summing over", then write the DP. Brute-force enumeration only runs for very small T, so it's a verifier, not a trainer.

In [ ]:
def ctc_forward_brute_force(log_probs: np.ndarray, labels: list, blank: int = 0) -> float:
    """Brute-force CTC (for verification only; usable when T is small).
    Enumerate every path of length T, keep those that collapse to labels, and pool them with logsumexp.
    """
    from itertools import product
    T, V = log_probs.shape

    def collapse(path):
        """Drop blanks and merge repeated characters → equivalent to the CTC collapse."""
        result, prev = [], None
        for c in path:
            if c == blank:
                prev = None
            elif c != prev:
                result.append(c)
                prev = c
        return result

    log_p_list = []
    for path in product(range(V), repeat=T):
        if collapse(list(path)) == list(labels):
            lp = sum(log_probs[t, path[t]] for t in range(T))
            log_p_list.append(lp)

    if not log_p_list:
        return -1e30

    arr = np.array(log_p_list)
    m = arr.max()
    return float(m + np.log(np.exp(arr - m).sum()))


# quick self-check (at T=4 brute force finishes in < 1 second)
np.random.seed(42)
_lp4 = log_softmax(np.random.randn(4, VOCAB_SIZE))
_ref = ctc_forward_brute_force(_lp4, LABELS, blank=BLANK)
print(f"Brute-force reference value (T=4): {_ref:.4f}  → used to assert the DP is correct")

In [ ]:
try:
    lp = ctc_forward(log_probs, LABELS, blank=BLANK)
    print(f'log P(labels | frames) = {lp:.4f}')
    print(f'P(labels | frames)     = {np.exp(lp):.8f}')
    assert np.isfinite(lp), 'the forward algorithm should return a finite value'
    assert lp < 0, 'a log-probability should be negative'

    # strong check: compare against the brute-force reference (at T=6, 3^6=729 paths, enumerated in < 1 second)
    lp_ref = ctc_forward_brute_force(log_probs, LABELS, blank=BLANK)
    print(f'Brute-force reference  = {lp_ref:.4f}')
    assert np.isclose(lp, lp_ref, atol=1e-5), \
        f'❌ Does not match the reference: DP={lp:.4f}, brute={lp_ref:.4f} — check the recursion or the initial conditions'

    print('✅ CTC forward algorithm verified (matches brute force to 1e-5)')
except (NotImplementedError, TypeError):
    print('⬜ ctc_forward is not implemented yet — complete TODO steps 1–3')

## Complexity and intuition

In [ ]:
try:
    # show it: as T grows, the work grows linearly (not exponentially)
    print(f'{"T":>4}  {"S=2L+1":>6}  {"cells":>8}  {"log P":>8}')
    print('-' * 35)
    for T_test in [5, 10, 20, 50, 100]:
        _logits_t = np.random.randn(T_test, VOCAB_SIZE)
        _logits_t -= _logits_t.max(axis=1, keepdims=True)  # numerical stability
        lp_test = _logits_t - np.log(np.exp(_logits_t).sum(axis=1, keepdims=True))
        result = ctc_forward(lp_test, LABELS, blank=BLANK)
        S = 2*len(LABELS)+1
        print(f'{T_test:>4}  {S:>6}  {T_test*S:>8}  {result:>8.3f}')
except (NotImplementedError, TypeError):
    print('⬜ ctc_forward is not implemented yet — complete TODO steps 1–3 first')

## Wrap-up

| Concept | What to remember |
|---|---|
| Extended label l' | A blank between (and around) every label, length 2L+1 |
| Forward variable α[t][s] | A log-domain recursion that avoids underflow |
| Three predecessors | Stay, move 1, jump 2 (only when the characters differ) |
| Complexity | O(T·S) — where `S = 2L + 1`; far faster than the exponential cost of enumerating paths |
| L70 | Whisper decodes with attention, but it is still a token-by-token autoregressive process |

The next lesson (L70) takes Whisper's architecture apart at the code level: the complete implementation of the audio encoder, cross-attention and the text decoder. First connect this lesson's log-domain DP with L67's table-filling mindset, then go and look at Whisper.

## ✏️ Closed-book derivation check cell — the CTC forward recursion for α

**Rule: close every cell above and complete the derivations below from memory.**

**Question 1 — the α recursion**: the CTC forward variable $\alpha(t, s)$ satisfies the recursion:

$$\alpha(t, s) = \Big[\alpha(t-1, s) + \alpha(t-1, s-1) + \alpha(t-1, s-2) \cdot \mathbf{1}[\text{condition}]\Big] \cdot p(l'_s \mid x_t)$$

Write out the full condition $\mathbf{1}[\text{condition}]$ (it involves blank and how it relates to neighboring non-blank labels).

**Question 2 — the blank merging rule**:

| Path | Label after collapsing |
|------|-----------|
| a - a | |
| a - ε - a | |
| a - ε - b | |

(ε = the blank symbol)

**Question 3 — the extended label**: for the label sequence $l = [a, b]$, write out $l'$ (with blanks) = ?

(Write your derivation here...)

In [ ]:
# Verify: numerical consistency against a precomputed reference value (fixed seed=42)
import numpy as np

# small example: vocab={blank=0, a=1, b=2}, label l=[a,b]
# extended label l' = [ε, a, ε, b, ε]
T, vocab_size = 5, 3
np.random.seed(42)
log_probs_raw = np.random.dirichlet(np.ones(vocab_size), size=T)
log_probs = np.log(log_probs_raw + 1e-9)  # (T, vocab)

l_prime = [0, 1, 0, 2, 0]  # [blank, a, blank, b, blank]
S = len(l_prime)

# forward recursion (reference implementation — do not modify)
NEG_INF = float('-inf')
log_alpha = np.full((T, S), NEG_INF)
log_alpha[0][0] = log_probs[0][l_prime[0]]
if S > 1: log_alpha[0][1] = log_probs[0][l_prime[1]]

for t in range(1, T):
    for s in range(S):
        paths = [log_alpha[t-1][s]]
        if s > 0:
            paths.append(log_alpha[t-1][s-1])
        # third path: valid only when the current symbol isn't blank and differs from the previous non-blank label
        if s > 1 and l_prime[s] != 0 and l_prime[s] != l_prime[s-2]:
            paths.append(log_alpha[t-1][s-2])
        log_alpha[t][s] = np.logaddexp.reduce(paths) + log_probs[t][l_prime[s]]

# final log P("ab" | input) = logaddexp(α(T-1, S-1), α(T-1, S-2))
log_p = np.logaddexp(log_alpha[-1][-1], log_alpha[-1][-2])

# precomputed reference value (fixed under seed=42)
REF_LOG_P = -1.361933   # computed ahead of time with the reference implementation

assert not np.isnan(log_p), "log_p is NaN — the recursion may have a numerical problem"
assert log_p < 0, "a log-probability must be < 0"
assert abs(log_p - REF_LOG_P) < 1e-4, (
    f"log_p = {log_p:.6f}, reference = {REF_LOG_P:.6f}\n"
    f"difference = {abs(log_p - REF_LOG_P):.2e} (should be < 1e-4)\n"
    f"check that the three-path condition of the α recursion in Question 1 is right"
)
print(f"log P(ab | input) = {log_p:.6f}  reference = {REF_LOG_P:.6f}")
print("✅ CTC forward algorithm verified (matches the reference value)")

In [ ]:
# ── Hand check: 2 frames · 3-symbol vocabulary (blank=0) ─────────────────
# Setup: vocab = {0:blank, 1:'a', 2:'b'}, label sequence = [1] (i.e. 'a')
# T=2 frames; a tiny probability matrix so the forward probability can be worked out by hand
import numpy as np

# log-probability matrix, shape=(T, V) — simple values that are easy to compute by hand
log_probs = np.log(np.array([
    [0.5, 0.4, 0.1],   # t=0: blank=0.5, a=0.4, b=0.1
    [0.3, 0.6, 0.1],   # t=1: blank=0.3, a=0.6, b=0.1
]))
labels = [1]   # target = 'a'

# expected value by hand (default blank=0):
# extended label l' = [0(blank), 1(a), 0(blank)]
# valid paths (collapse to [a]):
#   blank→a (0,1): 0.5×0.6 = 0.30
#   a→blank (1,0): 0.4×0.3 = 0.12
#   a→a     (1,1): 0.4×0.6 = 0.24  ← repeated characters collapse into one: a valid path
# P(labels|probs) = 0.30 + 0.12 + 0.24 = 0.66
expected_log_p = np.log(0.66)

try:
    # call this lesson's ctc_forward (default blank=0, which matches this example)
    log_p = ctc_forward(log_probs, labels)   # blank=0 (default)

    np.testing.assert_allclose(log_p, expected_log_p, atol=0.05,
        err_msg=f"CTC forward probability doesn't match the hand value: got {np.exp(log_p):.4f}, expected ≈ 0.66")
    print(f"✅ CTC hand check passed: P(a|frame) = {np.exp(log_p):.4f} (expected ≈ 0.66)")
    print("   valid paths: blank→a(0.30), a→blank(0.12), a→a(0.24) (repeated characters collapse)")
except (NotImplementedError, TypeError):
    print('⬜ ctc_forward is not implemented yet — complete TODO steps 1–3, then run this hand check')

## Understanding path collapsing and double counting

**Why can a path contain repeated characters (like "aa"), while "a-blank-a" gets special treatment?**

Consider the case where the target label is `[a]` (a single character):

```
Extended label l' = [∅, a, ∅] (positions 0, 1, 2)

Possible paths and what they collapse to:
─────────────────────────────────────────────
Path (one symbol per frame)    Collapsed     Valid?
─────────────────────────────────────────────
∅ → ∅ → ∅                    []           no (no characters)
∅ → ∅ → a                    [a]          ✓
∅ → a → ∅                    [a]          ✓
∅ → a → a                    [a]          ✓ (aa collapses to a)
a → ∅ → ∅                    [a]          ✓
a → ∅ → a                    [a, a]       no (the blank keeps the two a's apart)
a → a → ∅                    [a]          ✓ (aa collapses to a)
a → a → a                    [a]          ✓ (aaa collapses to a)
─────────────────────────────────────────────
```

**Key observations**:
1. **Repeated characters (a → a) are valid**, and still collapse to `[a]`
2. **A blank in between is different (a → ∅ → a)**: the collapsing rule is "merge adjacent repeats, then drop blanks", so the blank keeps the two a's separate and the result is `[a, a]` — not a path for `[a]`

**So why add the condition `l'[s] ≠ l'[s-2]`?**

Its job is to make sure **the DP only counts paths that really collapse to the target**.

For example, consider the target `[a, a]` (two a's):
```
Extended label l' = [∅, a, ∅, a, ∅] (positions 0, 1, 2, 3, 4)
```

Now consider computing α[2][3] at t=2 (position 3, the second a). Its predecessors could be:
- α[1][3] (stay)
- α[1][2] (move 1)
- **α[1][1] (jump 2)?**

If we **wrongly allowed α[1][1] → α[2][3]** (jumping straight from the first a to the second), the path would be:
```
... → a → a    (the path emits 'a' on both frame 1 and frame 2)
```
which collapses to `[a]` (not the target `[a, a]`)!

The only legitimate way to reach the second a is **through** α[1][2] → α[2][3] (from the blank in between to the second a):
```
... → ∅ → a    (frame 1 sits on position 2 (blank), frame 2 on position 3 (a))
```
and that path really does collapse to two a's.

So the condition `l'[s] ≠ l'[s-2]` **forbids jumping straight from position s-2 to position s when both positions hold the same character**. That keeps the DP from counting paths that collapse to the wrong label.

In summary:
- The target label may contain repeated characters (an a followed by an a)
- Paths may come in all kinds of combinations (consecutive repeats, blanks in between, and so on)
- But in the DP recursion, "forbidding certain jumps" is what makes every path counted once, and only if it collapses to the target


In [ ]:
# ✏️ Self-review for this lesson
l69_review = {
    "exp_path_count_understood": None,  # Do you understand why naive enumeration is exponential (V^T paths)? True/False
    "expanded_label_l_prime":    None,  # Do you remember l'=[∅,a,∅,b,∅], length S=2L+1? True/False
    "forward_var_recursion":     None,  # Can you write the α[t][s] recursion from memory (case 1/2/initialization)? True/False
    "ctc_forward_impl":          None,  # Is ctc_forward() correct, within 1e-5 of brute force? True/False
    "whiteboard_passed":         None,  # Did you clear the whiteboard challenge deriving the α recursion? True/False
}

unfilled = [k for k, v in l69_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l69_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L69 all clear! On to L70: Inside Whisper')

---

→ **Next lesson** · [L70 · Inside Whisper](L70_whisper_arch.ipynb)

> Next lesson covers **inside Whisper**: log-mel input, the Transformer encoder–decoder and the multitask heads. First connect this lesson's log-domain DP with L67's table-filling mindset, then go and read Whisper.